# Architecture 5: BioClinicalBERT + LoRA (Vitals-in-Text, No LightGBM)

**Changes from arch4_v1:**
1. **Vitals serialized into text** — BERT sees HR, BP, RR, SpO2, Temp inline with CC and HPI. Removes the artificial split between text and structured data.
2. **LoRA fine-tuning** — adapts all 12 BERT layers via low-rank adapters (~2M trainable params vs 21M in arch4). No manual layer unfreezing needed.
3. **LightGBM removed** — structured features are now encoded in text; no fusion branch needed. Simpler architecture.
4. **Single-phase training** — no head warmup / unfreeze dance; LoRA handles stable optimization from the start.

**Input:** `s3://ed-triage-capstone-group7/Data_Output/consolidated_dataset_features.csv`  
**Output:** `s3://ed-triage-capstone-group7/models/arch5/`

| Section | Content |
|---------|----------|
| 1 | Setup & installs |
| 2 | Imports & config |
| 3 | Load data from S3 |
| 4 | Text construction with vitals |
| 5 | Dataset & DataLoader |
| 6 | Model definition (LoRA) |
| 7 | Training loop |
| 8 | Training curves |
| 9 | Evaluation on test set |
| 10 | Save artifacts to S3 |

## 1. Setup & Installs

In [ ]:
!pip install -q boto3 transformers peft scikit-learn "numpy==1.26.4"

## 2. Imports & Config

In [ ]:
import io
import json
import math
import random
import warnings

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ── AWS / S3 Config ────────────────────────────────────────────────────────────
S3_BUCKET    = "ed-triage-capstone-group7"
DATA_KEY     = "Data_Output/consolidated_dataset_features.csv"
PMH_DATA_KEY = "Data_Output/consolidated_dataset_PMH_v2.csv"
SPLIT_PREFIX = "Data_Output/splits/arch5/"
MODEL_PREFIX = "models/arch5/"
BERT_MODEL   = "emilyalsentzer/Bio_ClinicalBERT"

# ── LoRA Config ────────────────────────────────────────────────────────────────
LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.1
LORA_TARGET_MODULES = ["query", "value"]  # standard for BERT

# ── Model / Training Hyperparameters ───────────────────────────────────────────
NUM_CLASSES        = 3
MAX_LEN            = 384
BATCH_SIZE         = 16
ACCUMULATION_STEPS = 2
EPOCHS             = 20
PATIENCE           = 4
BERT_LR            = 1e-4
HEAD_LR            = 2e-4
HEAD_HIDDEN_DIM    = 128
HEAD_DROPOUT       = 0.3

# ── Device ────────────────────────────────────────────────────────────────────
AMP_DTYPE       = torch.float16
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AUTOCAST_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
scaler          = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Load Data from S3

In [ ]:
# ── Debug mode — set True to run a cheap local sanity check ───────────────────
# Caps data to 300 rows, 2 epochs, skips S3 saves.
# Set False before running on SageMaker.
DEBUG_MODE = True

if DEBUG_MODE:
    EPOCHS             = 2
    PATIENCE           = 99   # don't early-stop in debug
    BATCH_SIZE         = 8
    ACCUMULATION_STEPS = 1
    print("DEBUG MODE ON — 300 rows, 2 epochs, no S3 saves")
else:
    print("PRODUCTION MODE — full data, S3 saves enabled")

In [ ]:
s3  = boto3.client("s3", region_name="us-east-1")
obj = s3.get_object(Bucket=S3_BUCKET, Key=DATA_KEY)
df  = pd.read_csv(io.BytesIO(obj["Body"].read()))

# ── Join patient_info (gender, race, age) from PMH dataset ────────────────────
pmh_obj = s3.get_object(Bucket=S3_BUCKET, Key=PMH_DATA_KEY)
df_pmh  = pd.read_csv(io.BytesIO(pmh_obj["Body"].read()), usecols=["stay_id", "patient_info"])
df      = df.merge(df_pmh, on="stay_id", how="left")
print(f"patient_info non-null: {df['patient_info'].notna().sum()} / {len(df)}")
print(f"Sample patient_info:   {df['patient_info'].iloc[0]}")

if DEBUG_MODE:
    df = df.groupby("triage", group_keys=False).apply(
        lambda x: x.sample(min(len(x), max(1, int(300 * len(x) / len(df)))), random_state=SEED)
    ).reset_index(drop=True)
    print(f"DEBUG: sampled {len(df)} rows")

print(f"\nLoaded: {df.shape}")
print(f"\nTriage distribution (raw 4-class):")
print(df["triage"].value_counts().sort_index().rename(
    {1: "L1-Critical", 2: "L2-Emergent", 3: "L3-Urgent", 4: "L4-LessUrgent"}
).to_string())

## 4. Text Construction with Vitals

**Key change from arch4:** Vitals are serialized inline into the text input.
BERT now sees the full clinical picture — `"Chief complaint: chest pain. Vitals: HR 140, BP 88/60, RR 22, SpO2 94%, Temp 98.6F. History: ..."`

This lets BERT learn interactions between symptoms and vitals that were invisible to it in arch4
(e.g. "shortness of breath + SpO2 88%" vs "shortness of breath + SpO2 99%").
HPI is trimmed slightly to 128 words to keep total length within MAX_LEN=384 tokens.

In [ ]:
TRANSPORT_LABEL = {"WALK IN": "walk-in", "AMBULANCE": "ambulance", "HELICOPTER": "helicopter", "UNKNOWN": "unknown"}

def clip_words(text, max_words):
    text = "" if pd.isna(text) else str(text).replace("\n", " ").strip()
    return " ".join(text.split()[:max_words]) if text else ""


def fmt_vital(val, fmt, label):
    if pd.isna(val):
        return ""
    return f"{label} {val:{fmt}}"


def build_triage_text(row):
    cc           = clip_words(row["chiefcomplaint"], 24)
    hpi          = clip_words(row["HPI"], 128)
    patient_info = "" if pd.isna(row["patient_info"]) else str(row["patient_info"]).strip()

    # Vitals — only include if present
    vital_parts = [
        fmt_vital(row["heart_rate"], ".0f", "HR"),
        fmt_vital(row["sbp"],        ".0f", "BP") + (
            f"/{row['dbp']:.0f}" if pd.notna(row["sbp"]) and pd.notna(row["dbp"]) else ""
        ),
        fmt_vital(row["resp_rate"],  ".0f", "RR"),
        fmt_vital(row["spo2"],       ".0f", "SpO2") + ("%" if pd.notna(row["spo2"]) else ""),
        fmt_vital(row["temp_f"],     ".1f", "Temp") + ("F" if pd.notna(row["temp_f"]) else ""),
    ]
    vitals_str = ", ".join(v for v in vital_parts if v)

    pain_str = ""
    if not row["pain_missing"] and pd.notna(row["pain"]):
        pain_str = f"Pain {row['pain']:.0f}/10"

    transport_str = TRANSPORT_LABEL.get(str(row["arrival_transport"]), "")

    parts = []
    if cc:            parts.append(f"Chief complaint: {cc}.")
    if cc:            parts.append(f"Presenting with {cc}.")  # CC emphasis
    if patient_info:  parts.append(f"Patient: {patient_info}.")
    if vitals_str:    parts.append(f"Vitals: {vitals_str}.")
    if pain_str:      parts.append(f"{pain_str}.")
    if transport_str: parts.append(f"Arrival: {transport_str}.")
    if hpi:           parts.append(f"History: {hpi}.")
    return " ".join(parts)


# 4-class → 3-class collapse (L4 merged into L3)
target_map = {1: 0, 2: 1, 3: 2, 4: 2}
df["triage_3class"] = df["triage"].map(target_map).astype(int)
df["triage_text"]   = df.apply(build_triage_text, axis=1)

CLASS_NAMES = ["L1-Critical", "L2-Emergent", "L3-Urgent/LessUrgent"]

print("3-class distribution:")
print(df["triage_3class"].value_counts().sort_index().rename(
    {0: CLASS_NAMES[0], 1: CLASS_NAMES[1], 2: CLASS_NAMES[2]}
).to_string())
print()
wc = df["triage_text"].str.split().str.len()
print(f"triage_text — 50th percentile: {wc.quantile(0.50):.0f} words")
print(f"triage_text — 90th percentile: {wc.quantile(0.90):.0f} words")
print(f"\nSample text (first row):")
print(df["triage_text"].iloc[0][:500] + "...")

In [ ]:
BASE_COLUMNS = [
    "stay_id", "triage", "triage_3class", "triage_text",
    "chiefcomplaint", "HPI", "arrival_transport",
    "pain", "pain_missing", "age",
    "temp_f", "heart_rate", "resp_rate", "spo2", "sbp", "dbp",
    "patient_info",  # gender, race, age string
]

X = df[BASE_COLUMNS].copy()
y = df["triage_3class"].values

# ── Stratified 80 / 10 / 10 split ─────────────────────────────────────────────
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)

total = len(df)
print(f"Train: {len(X_train)} ({len(X_train)/total:.1%})")
print(f"Val:   {len(X_val)} ({len(X_val)/total:.1%})")
print(f"Test:  {len(X_test)} ({len(X_test)/total:.1%})")
print()
print("Train class counts:")
print(pd.Series(y_train).value_counts().sort_index().rename(
    {0: CLASS_NAMES[0], 1: CLASS_NAMES[1], 2: CLASS_NAMES[2]}
).to_string())

if not DEBUG_MODE:
    def save_split_to_s3(split_df, y_split, key):
        out = split_df.copy()
        out["triage_3class"] = y_split
        buf = io.BytesIO()
        out.to_csv(buf, index=False)
        buf.seek(0)
        s3.put_object(Bucket=S3_BUCKET, Key=key, Body=buf.getvalue(), ContentType="text/csv")
        print(f"  Saved → s3://{S3_BUCKET}/{key}  ({len(out)} rows)")

    print("\nSaving splits:")
    save_split_to_s3(X_train, y_train, f"{SPLIT_PREFIX}train.csv")
    save_split_to_s3(X_val,   y_val,   f"{SPLIT_PREFIX}val.csv")
    save_split_to_s3(X_test,  y_test,  f"{SPLIT_PREFIX}test.csv")
else:
    print("\nDEBUG: skipping S3 split saves")

## 5. Dataset & DataLoader

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)


class TriageDataset(Dataset):
    """Tokenized triage_text + label. No LightGBM probs needed."""

    def __init__(self, frame, labels, tokenizer, max_len):
        self.texts     = frame["triage_text"].tolist()
        self.labels    = torch.tensor(labels, dtype=torch.long)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          self.labels[idx],
        }


train_dataset = TriageDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset   = TriageDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_dataset  = TriageDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

batch = next(iter(train_loader))
print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")
print(f"Effective batch size: {BATCH_SIZE * ACCUMULATION_STEPS}")
print({k: tuple(v.shape) for k, v in batch.items()})

## 6. Model Definition

`LoRATriageModel`: BioClinicalBERT with LoRA adapters on query+value in all 12 layers (~2M trainable params)  
→ mean pool → LayerNorm → Linear(768, 128) → GELU → Dropout(0.3) → Linear(128, 3)

No LightGBM fusion branch — vitals are now encoded in the text input.

In [ ]:
class LoRATriageModel(nn.Module):
    def __init__(self, bert_model_name, lora_config, num_classes=3,
                 hidden_dim=HEAD_HIDDEN_DIM, dropout=HEAD_DROPOUT):
        super().__init__()
        base_bert  = AutoModel.from_pretrained(bert_model_name)
        self.bert  = get_peft_model(base_bert, lora_config)
        bert_dim   = self.bert.config.hidden_size  # 768
        self.head  = nn.Sequential(
            nn.LayerNorm(bert_dim),
            nn.Linear(bert_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def masked_mean_pool(self, last_hidden_state, attention_mask):
        mask   = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden_state * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-6)
        return summed / counts

    def forward(self, input_ids, attention_mask):
        out    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.masked_mean_pool(out.last_hidden_state, attention_mask)
        return self.head(pooled)


lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
)

model = LoRATriageModel(BERT_MODEL, lora_config).to(DEVICE)
model.bert.gradient_checkpointing_enable()

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Trainable %:      {100 * trainable_params / total_params:.1f}%")
model.bert.print_trainable_parameters()

with torch.no_grad():
    logits = model(
        batch["input_ids"].to(DEVICE),
        batch["attention_mask"].to(DEVICE),
    )
    print(f"Forward pass OK — logits shape: {tuple(logits.shape)}")

## 7. Training Loop

Single-phase training — LoRA adapters + head trained together from epoch 1.  
No layer unfreezing schedule needed.

In [ ]:
# ── Class weights (sqrt-dampened inverse frequency) ────────────────────────────
train_counts = {int(c): int(n) for c, n in zip(*np.unique(y_train, return_counts=True))}
raw_class_weights = torch.tensor(
    [len(y_train) / (NUM_CLASSES * train_counts[c]) for c in range(NUM_CLASSES)],
    dtype=torch.float32,
)
CLASS_WEIGHTS = torch.sqrt(raw_class_weights)
print(f"Class weights (sqrt-dampened): {[round(x, 4) for x in CLASS_WEIGHTS.tolist()]}")

criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(DEVICE))

# ── Optimizer — separate LRs for LoRA adapters vs classification head ──────────
optimizer = AdamW([
    {"params": model.bert.parameters(), "lr": BERT_LR,  "weight_decay": 0.01},
    {"params": model.head.parameters(), "lr": HEAD_LR,  "weight_decay": 0.01},
])

steps_per_epoch = math.ceil(len(train_loader) / ACCUMULATION_STEPS)
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = max(1, int(0.10 * total_steps))
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
print(f"Total steps: {total_steps} | Warmup: {warmup_steps}")

In [ ]:
def compute_critical_f1(labels, preds):
    report = classification_report(labels, preds, labels=[0, 1, 2], output_dict=True, zero_division=0)
    return float(report["0"]["f1-score"])


def train_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0.0
    preds, labels = [], []
    optimizer.zero_grad()

    for step, batch in enumerate(loader, start=1):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        target         = batch["label"].to(DEVICE)

        with torch.amp.autocast(device_type=AUTOCAST_DEVICE, dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, target) / ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if step % ACCUMULATION_STEPS == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * ACCUMULATION_STEPS
        preds.extend(logits.detach().argmax(dim=1).cpu().numpy())
        labels.extend(target.detach().cpu().numpy())

    return {
        "loss":     total_loss / len(loader),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    preds, labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            target         = batch["label"].to(DEVICE)

            with torch.amp.autocast(device_type=AUTOCAST_DEVICE, dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
                logits = model(input_ids, attention_mask)
                loss   = criterion(logits, target)

            total_loss += loss.item()
            preds.extend(logits.detach().argmax(dim=1).cpu().numpy())
            labels.extend(target.detach().cpu().numpy())

    return {
        "loss":        total_loss / len(loader),
        "macro_f1":    f1_score(labels, preds, average="macro"),
        "critical_f1": compute_critical_f1(labels, preds),
        "preds":       np.array(preds),
        "labels":      np.array(labels),
    }


def predict_probs(model, loader):
    model.eval()
    outputs = []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            with torch.amp.autocast(device_type=AUTOCAST_DEVICE, dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
                logits = model(input_ids, attention_mask)
            outputs.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    return np.vstack(outputs)

In [ ]:
best_val_f1     = -1.0
bad_epochs      = 0
history         = []
best_model_path = "/tmp/best_model_arch5.pt"

for epoch in range(1, EPOCHS + 1):
    train_metrics = train_epoch(model, train_loader, optimizer, scheduler, criterion)
    val_metrics   = eval_epoch(model, val_loader, criterion)

    gap      = train_metrics["macro_f1"] - val_metrics["macro_f1"]
    improved = val_metrics["macro_f1"] > best_val_f1

    if improved:
        best_val_f1 = val_metrics["macro_f1"]
        bad_epochs  = 0
        torch.save(model.state_dict(), best_model_path)
        status = "<- best"
    else:
        bad_epochs += 1
        status = f"patience {bad_epochs}/{PATIENCE}"

    history.append({
        "epoch":           epoch,
        "train_loss":      train_metrics["loss"],
        "val_loss":        val_metrics["loss"],
        "train_f1":        train_metrics["macro_f1"],
        "val_f1":          val_metrics["macro_f1"],
        "val_critical_f1": val_metrics["critical_f1"],
        "generalization_gap": gap,
    })

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_metrics['loss']:.4f} train_f1={train_metrics['macro_f1']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} val_f1={val_metrics['macro_f1']:.4f} | "
        f"critical_f1={val_metrics['critical_f1']:.4f} gap={gap:.4f} | {status}"
    )

    if bad_epochs >= PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}.")
        break

print()
print(f"Best validation macro-F1: {best_val_f1:.4f}")
history_df = pd.DataFrame(history)
history_df

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train_loss")
axes[0].plot(history_df["epoch"], history_df["val_loss"],   marker="o", label="val_loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history_df["epoch"], history_df["train_f1"], marker="o", label="train_f1")
axes[1].plot(history_df["epoch"], history_df["val_f1"],   marker="o", label="val_f1")
axes[1].set_title("Macro-F1")
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(history_df["epoch"], history_df["generalization_gap"], marker="o",
             color="purple", label="train_f1 - val_f1")
axes[2].plot(history_df["epoch"], history_df["val_critical_f1"],    marker="o",
             color="crimson", label="val L1-Critical F1")
axes[2].set_title("Generalization gap & Critical F1")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle("Architecture 5: BioClinicalBERT + LoRA (Vitals-in-Text)", fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Evaluation on Test Set

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

val_metrics    = eval_epoch(model, val_loader,  criterion)
test_metrics   = eval_epoch(model, test_loader, criterion)
all_test_probs = predict_probs(model, test_loader)

weighted_f1 = f1_score(y_test, test_metrics["preds"], average="weighted")
roc_auc     = roc_auc_score(y_test, all_test_probs, multi_class="ovr", average="macro")

print("Validation macro-F1:   ", round(val_metrics["macro_f1"],  4))
print("Test macro-F1:         ", round(test_metrics["macro_f1"], 4))
print("Validation critical F1:", round(val_metrics["critical_f1"],  4))
print("Test critical F1:      ", round(test_metrics["critical_f1"], 4))
print("Test weighted F1:      ", round(weighted_f1, 4))
print("Test ROC-AUC (OvR):    ", round(roc_auc, 4))
print()

# ── Comparison vs arch4_v1 baseline ───────────────────────────────────────────
ARCH4_V1 = {"macro_f1": 0.6661, "critical_f1": 0.5750, "roc_auc": 0.8277}
print("--- vs arch4_v1 baseline ---")
print(f"  macro-F1:    {test_metrics['macro_f1']:.4f} vs {ARCH4_V1['macro_f1']:.4f} "
      f"(delta={test_metrics['macro_f1'] - ARCH4_V1['macro_f1']:+.4f})")
print(f"  critical-F1: {test_metrics['critical_f1']:.4f} vs {ARCH4_V1['critical_f1']:.4f} "
      f"(delta={test_metrics['critical_f1'] - ARCH4_V1['critical_f1']:+.4f})")
print(f"  ROC-AUC:     {roc_auc:.4f} vs {ARCH4_V1['roc_auc']:.4f} "
      f"(delta={roc_auc - ARCH4_V1['roc_auc']:+.4f})")
print()

print("Classification report:")
print(classification_report(
    y_test, test_metrics["preds"],
    target_names=CLASS_NAMES, zero_division=0
))

summary_df = pd.DataFrame({
    "metric": [
        "val_macro_f1", "test_macro_f1",
        "val_critical_f1", "test_critical_f1",
        "test_weighted_f1", "test_roc_auc_ovr_macro",
    ],
    "value": [
        val_metrics["macro_f1"], test_metrics["macro_f1"],
        val_metrics["critical_f1"], test_metrics["critical_f1"],
        weighted_f1, roc_auc,
    ],
})
summary_df

In [ ]:
cm     = confusion_matrix(y_test, test_metrics["preds"], labels=[0, 1, 2])
cm_pct = cm / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm_pct, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=["Critical", "Emergent", "Urgent/LU"],
    yticklabels=["Critical", "Emergent", "Urgent/LU"],
)
plt.title(
    "Confusion Matrix (% of actual class)\n"
    "Arch5: BioClinicalBERT + LoRA (Vitals-in-Text)",
    fontweight="bold"
)
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.show()

print("\nRaw confusion matrix:")
print(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES))

## 10. Save Artifacts to S3

In [ ]:
if DEBUG_MODE:
    print("DEBUG MODE — skipping S3 artifact saves.")
else:
    def upload_file(local_path, s3_key):
        s3.upload_file(local_path, S3_BUCKET, s3_key)
        print(f"  Uploaded → s3://{S3_BUCKET}/{s3_key}")

    print(f"Saving artifacts to s3://{S3_BUCKET}/{MODEL_PREFIX}")

    # 1. Model weights (LoRA adapters merged into base for portability)
    merged_model = model.bert.merge_and_unload()
    model_path   = "/tmp/arch5_best_model.pt"
    torch.save({
        "bert_state_dict": merged_model.state_dict(),
        "head_state_dict": model.head.state_dict(),
    }, model_path)
    upload_file(model_path, f"{MODEL_PREFIX}best_model.pt")

    # 2. Training history
    history_path = "/tmp/arch5_history.csv"
    history_df.to_csv(history_path, index=False)
    upload_file(history_path, f"{MODEL_PREFIX}history.csv")

    # 3. Config
    config = {
        "architecture":  "arch5",
        "description":   "BioClinicalBERT + LoRA, vitals serialized in text, no LightGBM",
        "bert_model":    BERT_MODEL,
        "max_len":       MAX_LEN,
        "lora_r":        LORA_R,
        "lora_alpha":    LORA_ALPHA,
        "lora_dropout":  LORA_DROPOUT,
        "lora_targets":  LORA_TARGET_MODULES,
        "n_classes":     NUM_CLASSES,
        "text_format":   "CC_2x + Vitals + Pain + Transport + HPI",
        "changes_vs_arch4_v1": [
            "Vitals serialized into text (HR, BP, RR, SpO2, Temp, Pain, Transport)",
            "LoRA fine-tuning of all 12 layers (~2M trainable params vs 21M in arch4)",
            "LightGBM removed — no fusion branch",
            "Single-phase training — no head warmup / layer unfreeze schedule",
        ],
        "test_metrics": {
            "macro_f1":    round(test_metrics["macro_f1"],  4),
            "critical_f1": round(test_metrics["critical_f1"], 4),
            "weighted_f1": round(weighted_f1, 4),
            "roc_auc_ovr": round(roc_auc, 4),
        },
        "best_val_f1": round(best_val_f1, 4),
    }
    config_path = "/tmp/arch5_config.json"
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)
    upload_file(config_path, f"{MODEL_PREFIX}config.json")

    print()
    print("All artifacts saved.")
    print(f"Config: {json.dumps(config, indent=2)}")